# Detector Uncertainties

- inspect the WireMod+calovar envelope, per calo parameter

- take max variation per parameter as the unisim uncertainty

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime

# local imports
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from makedf.geniesyst import *
from analysis_village.numucc_1p0pi.utils import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import pickle
from analysis_village.numucc_1p0pi.utils import *
from pyanalib.covariance import *
from pyanalib.variable_calculator import *

In [3]:
var_configs = [
    VariableConfig.all_events(),
    # VariableConfig.vertex_x(),
    # VariableConfig.vertex_y(),
    # VariableConfig.vertex_z(),
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    # VariableConfig.muon_direction_x(),
    # VariableConfig.muon_direction_y(),
    VariableConfig.proton_momentum(),
    VariableConfig.proton_direction(),
    # VariableConfig.proton_direction_x(),
    # VariableConfig.proton_direction_y(),
    # VariableConfig.opening_angle(),
    VariableConfig.tki_del_Tp(),
    VariableConfig.tki_del_Tp_x(),
    VariableConfig.tki_del_Tp_y(),
    VariableConfig.tki_del_p(),
    VariableConfig.tki_del_alpha(),
    VariableConfig.tki_del_phi(),
    ]

In [4]:
save_fig = False
show_plot = True

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics_studies_detvar-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)


# file with hist counts from running detvar samples through event selection
sample_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10/nevts"

In [5]:
syst_type = "detector"
detector_dict = {}
detector_dict[syst_type] = {}
detector_dict[syst_type+"-wiremod_xtxw"] = {}
detector_dict[syst_type+"-wiremod_yz"] = {}
detector_dict[syst_type+"-0xSCE"] = {}
detector_dict[syst_type+"-2xSCE"] = {}

## WireMod + calo

### All combinations

In [ ]:
names = []
for ccal_var in ["p", "cv", "m"]:
    for alpha_var in ["p", "cv", "m"]:
        for beta_var in ["p", "cv", "m"]:
            for R_var in ["p", "cv", "m"]:
                names.append(f"evt_ccal_{ccal_var}-alpha_{alpha_var}-beta_{beta_var}-R_{R_var}")


In [ ]:
ret_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags=["wiremod_bnb_20260211_yz_updatecalo_allvars_sel_2prong_vtxdist_test"]+["wiremod_bnb_20260211_yz_updatecalo_allvars_sel_2prong_vtxdist_"+t for t in generate_tags("ai")[2:]],
    keys2load=names+["meta"],
    n_max_concat=999
)

In [ ]:
ret_dfs.keys()

In [ ]:
mc_meta_df = ret_dfs['meta']

for k in ret_dfs.keys():
    if "meta" in k:
        continue
    ret_dfs[k].loc[:,'topo_categ'] = get_topo_category(ret_dfs[k])
    ret_dfs[k].loc[:,'genie_categ'] = get_genie_category(ret_dfs[k])

In [ ]:
for k in ret_dfs.keys():
    if "meta" in k:
        continue

    for trk_idx in [1,2]:
        for part in ["muon", "proton"]:
            chimu_avg = avg_chi2(ret_dfs[k]["trk{}".format(trk_idx)], "chi2_{}_new".format(part))
            ret_dfs[k][("trk{}".format(trk_idx), "pfp", "trk", "chi2pid", "avg", "chi2_{}_new".format(part), "")] = chimu_avg

In [ ]:
var_config = VariableConfig.chi2_mu()
bins = var_config.bins

ptag = "avg"
for kidx,k in enumerate(ret_dfs.keys()):
    if "meta" in k:
        continue
    this_df = ret_dfs[k]
    var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
    var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
    if kidx == 0:
        plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="C0", alpha=0.5, density=True, label="WireMod+Calo Variations") #, color=var_plot_dict[param][1], linestyle=linestyles[didx], linewidth=2, density=True)
    plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="C0", alpha=0.5, density=True) #, color=var_plot_dict[param][1], linestyle=linestyles[didx], linewidth=2, density=True)

cv_key = "evt_ccal_cv-alpha_cv-beta_cv-R_cv"
this_df = ret_dfs[cv_key]
var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="black", linewidth=2, density=True, label="CV")

# plt.axvline(25, color="red", linestyle="--")

plt.xlim(bins[0], bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.legend()
plt.savefig(path.join(save_fig_dir, "chi2_muon_wiremod_calo_fullvars.png"), dpi=300)
plt.show();

In [ ]:
var_config = VariableConfig.chi2_proton()
bins = var_config.bins

ptag = "avg"
for kidx,k in enumerate(ret_dfs.keys()):
    if "meta" in k:
        continue
    this_df = ret_dfs[k]
    var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_proton_new", "")]
    var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_proton_new", "")]
    if kidx == 0:
        plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="C0", alpha=0.5, density=True, label="WireMod+Calo Variations") #, color=var_plot_dict[param][1], linestyle=linestyles[didx], linewidth=2, density=True)
    plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="C0", alpha=0.5, density=True) #, color=var_plot_dict[param][1], linestyle=linestyles[didx], linewidth=2, density=True)

cv_key = "evt_ccal_cv-alpha_cv-beta_cv-R_cv"
this_df = ret_dfs[cv_key]
var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_proton_new", "")]
var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_proton_new", "")]
plt.hist(pd.concat([var_1, var_2]), bins=bins, histtype="step", color="black", linewidth=2, density=True, label="CV") 

# plt.axvline(90, color="red", linestyle="--")

plt.xlim(bins[0], bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.legend()
plt.savefig(path.join(save_fig_dir, "chi2_proton_wiremod_calo_fullvars.png"), dpi=300)
plt.show();

### individual parameter variations on 2-prong selected slices

In [6]:
# WireMod Variations
keys2load=['meta', 'evt_cv', 'evt_ccal_p', 'evt_ccal_m', 'evt_beta_p', 'evt_beta_m', 'evt_alpha_p', 'evt_alpha_m', 'evt_R_p', 'evt_R_m']

ret_XThetaXW_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags = ["wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_aa",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ab",
                  # "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ac",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ad",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ae",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_af",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ag",
                  "wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ah"],
    keys2load=keys2load,
    n_max_concat=999
)

ret_YZ_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags = ["wiremod_bnb_20260211_yz_updatecalo_sel_2prong_aa",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ab",
                  # "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ac",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ad",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ae",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_af",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ag",
                  "wiremod_bnb_20260211_yz_updatecalo_sel_2prong_ah"],
    keys2load=keys2load,
    n_max_concat=999
)

Keys: ['/evt_R_m_0', '/evt_R_p_0', '/evt_alpha_m_0', '/evt_alpha_p_0', '/evt_beta_m_0', '/evt_beta_p_0', '/evt_ccal_m_0', '/evt_ccal_p_0', '/evt_cv_0', '/histgenevtdf_0', '/histpotdf_0', '/meta_0', '/split']
Reading file with tag wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_aa, mc_n_split: 1
Keys: ['/evt_R_m_0', '/evt_R_p_0', '/evt_alpha_m_0', '/evt_alpha_p_0', '/evt_beta_m_0', '/evt_beta_p_0', '/evt_ccal_m_0', '/evt_ccal_p_0', '/evt_cv_0', '/histgenevtdf_0', '/histpotdf_0', '/meta_0', '/split']
Reading file with tag wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ab, mc_n_split: 1
Keys: ['/evt_R_m_0', '/evt_R_p_0', '/evt_alpha_m_0', '/evt_alpha_p_0', '/evt_beta_m_0', '/evt_beta_p_0', '/evt_ccal_m_0', '/evt_ccal_p_0', '/evt_cv_0', '/histgenevtdf_0', '/histpotdf_0', '/meta_0', '/split']
Reading file with tag wiremod_bnb_20260211_xtxw_updatecalo_sel_2prong_ad, mc_n_split: 1
Keys: ['/evt_R_m_0', '/evt_R_p_0', '/evt_alpha_m_0', '/evt_alpha_p_0', '/evt_beta_m_0', '/evt_beta_p_0', '/evt_c

In [7]:
for k in keys2load:
    if "meta" in k:
        continue

    for trk_idx in [1,2]:
        for part in ["muon", "proton"]:
            chimu_avg = avg_chi2(ret_XThetaXW_dfs[k]["trk{}".format(trk_idx)], "chi2_{}_new".format(part))
            ret_XThetaXW_dfs[k][("trk{}".format(trk_idx), "pfp", "trk", "chi2pid", "avg", "chi2_{}_new".format(part), "")] = chimu_avg

In [ ]:
from analysis_village.numucc_1p0pi.makedf.selections import *

var_config = VariableConfig.track_score()

var_plot_dict = {
    "ccal": ["$C_{cal}$", "C0"],
    "alpha": ["$\\alpha$", "C1"],
    "beta": ["$\\beta$", "C2"],
    "R": ["$R$", "C3"],
}
linestyles = ["--", ":"] # for + and - variations
sign_texts = [" (+1$\\sigma$)", " (-1$\\sigma$)"]

ptag = "avg"

this_df = ret_XThetaXW_dfs["evt_cv"]
this_df = cut_2prong_contained(this_df, det=DETECTOR)
# this_df = cut_2prong_trackscore(this_df, trackscore_th=TRACKSCORE_TH)
var_1 = this_df.trk1[var_config.var_evt_reco_col]
var_2 = this_df.trk2[var_config.var_evt_reco_col]
n_xtxw, bins, _ = plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", label="X-Theta-XW", color="C0", linewidth=2, density=True)

this_df = ret_YZ_dfs["evt_cv"]
this_df = cut_2prong_contained(this_df, det=DETECTOR)
# this_df = cut_2prong_trackscore(this_df, trackscore_th=TRACKSCORE_TH)
var_1 = this_df.trk1[var_config.var_evt_reco_col]
var_2 = this_df.trk2[var_config.var_evt_reco_col]
n_yz, bins, _ = plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", label="Y-Z", color="C1", linewidth=2, density=True)

plt.legend()
plt.show()
print(n_xtxw/n_yz)

In [ ]:
var_config = VariableConfig.chi2_mu()

var_plot_dict = {
    "ccal": ["$C_{cal}$", "C0"],
    "alpha": ["$\\alpha$", "C1"],
    "beta": ["$\\beta$", "C2"],
    "R": ["$R$", "C3"],
}
linestyles = ["--", ":"] # for + and - variations
sign_texts = [" (+1$\\sigma$)", " (-1$\\sigma$)"]

ptag = "avg"

this_df = ret_XThetaXW_dfs["evt_cv"]
this_df = cut_2prong_contained(this_df, det=DETECTOR)
this_df = cut_2prong_trackscore(this_df, trackscore_th=TRACKSCORE_TH)
this_df = cut_2prong_vtxdist(this_df, vtxdist_th=VTXDIST_TH)
var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", label="CV", color="black", linewidth=2, density=True)

for param in ["ccal", "alpha", "beta", "R"]:
    for didx, detvar_name in enumerate([param+"_p", param+"_m"]):
        this_df = ret_XThetaXW_dfs["evt_{}".format(detvar_name)]
        this_df = cut_2prong_contained(this_df, det=DETECTOR)
        this_df = cut_2prong_trackscore(this_df, trackscore_th=TRACKSCORE_TH)
        this_df = cut_2prong_vtxdist(this_df, vtxdist_th=VTXDIST_TH)
        var_1 = this_df.trk1[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
        var_2 = this_df.trk2[("pfp", "trk", "chi2pid", ptag, "chi2_muon_new", "")]
        plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", color=var_plot_dict[param][1], linestyle=linestyles[didx], linewidth=2, density=True)

# legends
for k in var_plot_dict.keys():
    plt.hist([], histtype="step", color=var_plot_dict[k][1], label=var_plot_dict[k][0], linewidth=2, density=True)

for lidx, l in enumerate(linestyles):
    plt.hist([], histtype="step", color="gray", linestyle=l, label=sign_texts[lidx], linewidth=2, density=True)

plt.legend()
plt.show()

### Variations on the selected event rate

In [12]:
# CV
ret_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/MC/BNB_cosmics/updatecalo",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags=generate_tags("bl"),
    keys2load=['meta', 'evt'],
    n_max_concat=999
)

cv_mc_df = ret_dfs["evt"]
cv_meta_df = ret_dfs["meta"]

cv_mc_df.loc[:,'topo_categ'] = get_topo_category(cv_mc_df)
cv_mc_df.loc[:,'genie_categ'] = get_genie_category(cv_mc_df)

Keys: ['/evt_0', '/hdr_0', '/histgenevtdf_0', '/histpotdf_0', '/meta_0', '/split']
Reading file with tag aa, mc_n_split: 1
Keys: ['/evt_0', '/hdr_0', '/histgenevtdf_0', '/histpotdf_0', '/meta_0', '/split']
Reading file with tag ab, mc_n_split: 1


KeyboardInterrupt: 

In [ ]:
# WireMod Variations
ret_XThetaXW_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags = ["wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_aa",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ab",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ac",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ad",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ae",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_af",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ag",
                  "wiremod_bnb_20260211_xtxw_updatecalo_vars_sel_mup_ah"],
    keys2load=['meta', 'evt_ccal_p', 'evt_ccal_m', 'evt_beta_p', 'evt_beta_m', 'evt_alpha_p', 'evt_alpha_m', 'evt_R_p', 'evt_R_m'],
    n_max_concat=999
)

ret_YZ_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags = ["wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_aa",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ab",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ac",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ad",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ae",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_af",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ag",
                  "wiremod_bnb_20260211_yz_updatecalo_vars_sel_mup_ah"],
    keys2load=['meta', 'evt_ccal_p', 'evt_ccal_m', 'evt_beta_p', 'evt_beta_m', 'evt_alpha_p', 'evt_alpha_m', 'evt_R_p', 'evt_R_m'],
    n_max_concat=999
)

In [ ]:
# select events that exist in all detvar samples

cv_idx_df = cv_meta_df.reset_index().set_index(["run", "subrun", "evt", "E"])
meta_XThetaXW_idx_df = ret_XThetaXW_dfs["meta"].reset_index().set_index(["run", "subrun", "evt", "E"])
meta_YZ_idx_df = ret_YZ_dfs["meta"].reset_index().set_index(["run", "subrun", "evt", "E"])

cv_idx_list = cv_idx_df.index
common_idx_list = [idx for idx in cv_idx_list if idx in meta_XThetaXW_idx_df.index and idx in meta_YZ_idx_df.index]
# common_idx_list = [idx for idx in meta_XThetaXW_idx_df.index if idx in meta_YZ_idx_df.index]

meta_XThetaXW_idx_df = meta_XThetaXW_idx_df.loc[common_idx_list]
ret_XThetaXW_dfs["meta"] = meta_XThetaXW_idx_df
meta_YZ_idx_df = meta_YZ_idx_df.loc[common_idx_list]
ret_YZ_dfs["meta"] = meta_YZ_idx_df

print(len(meta_XThetaXW_idx_df.index))
print(len(meta_YZ_idx_df.index))
print(len(common_idx_list))

In [ ]:
new_dfs = []
for detvar_dfs in [ret_dfs, ret_XThetaXW_dfs, ret_YZ_dfs]:
    for k in tqdm(detvar_dfs.keys()):
        if "meta" in k:
            continue

        detvar_dfs[k]["E"] = detvar_dfs[k].mc.E.copy()
        this_meta_df = detvar_dfs["meta"].copy()
        this_meta_df = this_meta_df.reset_index().set_index(["run", "subrun", "evt", "E"])
        this_meta_df = this_meta_df.loc[common_idx_list]
        this_common_idx = this_meta_df.reset_index().set_index(["__ntuple", "entry", "E"]).index
        this_sel_df = detvar_dfs[k].reset_index().set_index(["__ntuple", "entry", "E"]) # .loc[this_common_idx]

        this_sel_idx = this_sel_df.index
        this_sel_common_idx = [idx for idx in this_sel_idx if idx in this_common_idx]
        this_sel_df = this_sel_df.loc[this_sel_common_idx]
        this_sel_df = this_sel_df.reset_index().set_index(["__ntuple", "entry", "rec.slc..index"])

        # mc_tot_pot = detvar_dfs["meta"]['pot'].sum()
        # print("mc_tot_pot: %.3e" %(mc_tot_pot))
        # data_tot_pot = 4.57e18
        # mc_pot_scale = data_tot_pot / mc_tot_pot
        # this_sel_df["pot_weight"] = mc_pot_scale * np.ones(len(this_sel_df))

        detvar_dfs[k] = this_sel_df.groupby(level=[0,1]).head(1)

        # plt.hist(this_sel_df.mc.E, bins=bins, histtype="step", label=k)

    new_dfs.append(detvar_dfs)

In [ ]:
# check that the Enu distribution is the same for all detvar samples

for detvar_dfs in [ret_dfs, ret_XThetaXW_dfs, ret_YZ_dfs]:
# for detvar_dfs in [ret_XThetaXW_dfs, ret_YZ_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        bins = np.linspace(0,2,51)
        this_var = detvar_dfs[k].mc.E
        # this_var = detvar_dfs[k].reset_index().E

        # bins = np.linspace(0,80,41)
        # this_var = detvar_dfs[k].p.pfp.trk.chi2pid.I2.chi2_muon_new

        plt.hist(this_var, bins=bins, histtype="step", label=k)
plt.legend()
plt.show();

In [ ]:
tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

for detvar_dfs in [ret_dfs, ret_XThetaXW_dfs, ret_YZ_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        var_mc_df = detvar_dfs[k]

        # add tki variables
        slc_mudf = var_mc_df.mu.pfp.trk.truth.p
        slc_pdf = var_mc_df.p.pfp.trk.truth.p
        slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
        slc_P_p_col = pad_column_name(("totp",), slc_pdf)
        tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
        for var_name in tki_var_names:
            var_mc_df = multicol_add(var_mc_df, tki_reco[var_name].rename("mc_" + var_name))

In [ ]:
for var_config in var_configs:

    ret = signal_hists(evtdf=ret_dfs["evt"], var_config=var_config, save_fig=False, plot=False, return_data=True)
    n_cv = ret["nevts_allsel_reco"]

    for widx, detvar_dfs in enumerate([ret_XThetaXW_dfs, ret_YZ_dfs]):
        for k in detvar_dfs.keys():
            if "meta" in k:
                continue

            var_mc_df = detvar_dfs[k]
            var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
            var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)

            ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
            n_var = ret["nevts_allsel_reco"]
            n_var, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_var, 
                                histtype="step", color=f"C{widx}", alpha=0.3, linewidth=2)

    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_cv, histtype="step", color="black", label="CV")

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Events / Bin")

    plt.hist([], histtype="step", color="C0", label="WireMod X-$\\theta_{XW}$")
    plt.hist([], histtype="step", color="C1", label="WireMod Y-Z")
    plt.legend(frameon=False, loc="best")

    if save_fig:
        plt.savefig(f"{save_fig_dir}/{var_config.var_save_name}_detvar_evt_rates.png", bbox_inches='tight', dpi=300)
    plt.show()

    # get uncertainty
    wiremod_titles = ["WireMod X-$\\theta_{XW}$", "WireMod Y-Z"]
    wiremod_tags = ["wiremod_xtxw", "wiremod_yz"]
    tot_frac_unc = np.zeros(len(var_config.bin_centers))
    for widx, detvar_dfs in enumerate([ret_XThetaXW_dfs, ret_YZ_dfs]):

        this_var_univ = []
        for k in detvar_dfs.keys():
            if "meta" in k:
                continue

            var_mc_df = detvar_dfs[k]
            var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
            var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)

            ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
            n_var = ret["nevts_allsel_reco"]
            this_var_univ.append(n_var)

        var_diffs = [this_var_univ[i] - n_cv for i in range(len(this_var_univ))]

        max_arg_diff = np.zeros(len(var_config.bin_centers))
        for i in range(len(var_config.bin_centers)):
            this_element_vals = [np.abs(v[i]) for v in var_diffs]
            max_arg = np.argmax(this_element_vals)
            max_arg_diff[i] = var_diffs[max_arg][i] 
        max_arg_diff

        n_var = n_cv + max_arg_diff
        ret = get_covariance_matrix(np.array([n_var]), n_cv)

        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        tot_frac_unc = tot_frac_unc + frac_unc**2

        plot_labels = ["", "", wiremod_titles[widx]]
        save_fig_name = "{}/{}-{}-{}".format(save_fig_dir, var_config.var_save_name, wiremod_tags[widx], "frac_unc")
        plot_frac_unc([frac_unc], var_config, plot_labels=plot_labels, save_fig=save_fig, save_name=save_fig_name)

        # matrix_type = "cov"
        # save_fig_name = "{}/{}-{}-{}".format(save_fig_dir, var_config.var_save_name, syst_type, matrix_type)
        # title = "{} {}".format(syst_type, matrix_type)
        # plot_heatmap(ret[matrix_type], 
        #             var_config.bins, 
        #             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
        #             save_fig=save_fig, save_name=save_fig_name)

        detector_dict[f"detector-{wiremod_tags[widx]}"][var_config.var_save_name] = ret
    
    tot_frac_unc = np.sqrt(tot_frac_unc)
    print(tot_frac_unc)

## SCE

In [ ]:
# CV
ret_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags=["SystVar_CV_sel_2prong"],
    keys2load=['meta', 'evt'],
    n_max_concat=999
)

cv_mc_df = ret_dfs["evt"]
cv_meta_df = ret_dfs["meta"]

cv_mc_df.loc[:,'topo_categ'] = get_topo_category(cv_mc_df)
cv_mc_df.loc[:,'genie_categ'] = get_genie_category(cv_mc_df)

# SCE vars
ret_0xSCE_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags=["SystVar_0xSCE_sel_2prong"],
    keys2load=['meta', 'evt'],
    n_max_concat=999
)

_0xSCE_mc_df = ret_0xSCE_dfs["evt"]
_0xSCE_meta_df = ret_0xSCE_dfs["meta"]

_0xSCE_mc_df.loc[:,'topo_categ'] = get_topo_category(_0xSCE_mc_df)
_0xSCE_mc_df.loc[:,'genie_categ'] = get_genie_category(_0xSCE_mc_df)

ret_2xSCE_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    sub_dir="",
    sample_dir="",
    df_tag="",
    chunk_tags=["SystVar_2xSCE_sel_2prong"],
    keys2load=['meta', 'evt'],
    n_max_concat=999
)

_2xSCE_mc_df = ret_2xSCE_dfs["evt"]
_2xSCE_meta_df = ret_2xSCE_dfs["meta"]

_2xSCE_mc_df.loc[:,'topo_categ'] = get_topo_category(_2xSCE_mc_df)
_2xSCE_mc_df.loc[:,'genie_categ'] = get_genie_category(_2xSCE_mc_df)

In [ ]:
# select events that exist in all detvar samples

cv_idx_df = cv_meta_df.reset_index().set_index(["run", "subrun", "evt", "E"])
meta_0xSCE_idx_df = ret_0xSCE_dfs["meta"].reset_index().set_index(["run", "subrun", "evt", "E"])
meta_2xSCE_idx_df = ret_2xSCE_dfs["meta"].reset_index().set_index(["run", "subrun", "evt", "E"])

cv_idx_list = cv_idx_df.index
common_idx_list = [idx for idx in cv_idx_list if idx in meta_0xSCE_idx_df.index and idx in meta_2xSCE_idx_df.index]
# common_idx_list = [idx for idx in meta_0xSCE_idx_df.index if idx in meta_2xSCE_idx_df.index]

meta_0xSCE_idx_df = meta_0xSCE_idx_df.loc[common_idx_list]
ret_0xSCE_dfs["meta"] = meta_0xSCE_idx_df
meta_2xSCE_idx_df = meta_2xSCE_idx_df.loc[common_idx_list]
ret_2xSCE_dfs["meta"] = meta_2xSCE_idx_df

print(len(meta_0xSCE_idx_df.index))
print(len(meta_2xSCE_idx_df.index))
print(len(common_idx_list))

In [ ]:
new_dfs = []
for detvar_dfs in [ret_dfs, ret_0xSCE_dfs, ret_2xSCE_dfs]:
    for k in tqdm(detvar_dfs.keys()):
        if "meta" in k:
            continue

        detvar_dfs[k]["E"] = detvar_dfs[k].mc.E.copy()
        this_meta_df = detvar_dfs["meta"].copy()
        this_meta_df = this_meta_df.reset_index().set_index(["run", "subrun", "evt", "E"])
        this_meta_df = this_meta_df.loc[common_idx_list]
        this_common_idx = this_meta_df.reset_index().set_index(["__ntuple", "entry", "E"]).index
        this_sel_df = detvar_dfs[k].reset_index().set_index(["__ntuple", "entry", "E"]) # .loc[this_common_idx]

        this_sel_idx = this_sel_df.index
        this_sel_common_idx = [idx for idx in this_sel_idx if idx in this_common_idx]
        this_sel_df = this_sel_df.loc[this_sel_common_idx]
        this_sel_df = this_sel_df.reset_index().set_index(["__ntuple", "entry", "rec.slc..index"])

        # mc_tot_pot = detvar_dfs["meta"]['pot'].sum()
        # print("mc_tot_pot: %.3e" %(mc_tot_pot))
        # data_tot_pot = 4.57e18
        # mc_pot_scale = data_tot_pot / mc_tot_pot
        # this_sel_df["pot_weight"] = mc_pot_scale * np.ones(len(this_sel_df))

        detvar_dfs[k] = this_sel_df.groupby(level=[0,1]).head(1)

        # plt.hist(this_sel_df.mc.E, bins=bins, histtype="step", label=k)

    new_dfs.append(detvar_dfs)

In [ ]:
# check that the Enu distribution is the same for all detvar samples

for detvar_dfs in [ret_dfs, ret_0xSCE_dfs, ret_2xSCE_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        bins = np.linspace(0,2,51)
        this_var = detvar_dfs[k].mc.E
        # this_var = detvar_dfs[k].reset_index().E

        # bins = np.linspace(0,80,41)
        # this_var = detvar_dfs[k].p.pfp.trk.chi2pid.I2.chi2_muon_new

        plt.hist(this_var, bins=bins, histtype="step", label=k)
plt.legend()
plt.show();

In [ ]:
# for 2prong files, compare the selection variable distributions

var_config = VariableConfig.track_score()
for detvar_dfs in [ret_dfs, ret_0xSCE_dfs, ret_2xSCE_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        this_df = detvar_dfs[k]
        var_1 = this_df.trk1[var_config.var_evt_reco_col]
        var_2 = this_df.trk2[var_config.var_evt_reco_col]

        plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", density=True)
plt.show();

var_config = VariableConfig.trk_len()
for detvar_dfs in [ret_dfs, ret_0xSCE_dfs, ret_2xSCE_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        this_df = detvar_dfs[k]
        var_1 = this_df.trk1[var_config.var_evt_reco_col]
        var_2 = this_df.trk2[var_config.var_evt_reco_col]

        plt.hist(pd.concat([var_1, var_2]), bins=var_config.bins, histtype="step", density=True)
plt.show();

In [ ]:
with open("/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/analysis_village/numucc_1p0pi/scripts/start_pos_list-SCEvar.pkl", "rb") as f:
    save_dict = pickle.load(f)

start_x_list = save_dict["x"]
start_y_list = save_dict["y"]
start_z_list = save_dict["z"]

bins = np.linspace(-0.5, 0.5, 101)

diff_0x = np.array(start_z_list)[:, 0] - np.array(start_z_list)[:, 1]
diff_0x = np.clip(diff_0x, bins[0], bins[-1])
plt.hist(diff_0x, bins=bins, histtype="step", label="CV - 0xSCE")

diff_2x = np.array(start_z_list)[:, 0] - np.array(start_z_list)[:, 2]
diff_2x = np.clip(diff_2x, bins[0], bins[-1])
plt.hist(diff_2x, bins=bins, histtype="step", label="CV - 2xSCE")
plt.show()

In [ ]:
tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

for detvar_dfs in [ret_dfs, ret_0xSCE_dfs, ret_2xSCE_dfs]:
    for k in detvar_dfs.keys():
        if "meta" in k:
            continue

        var_mc_df = detvar_dfs[k]

        # add tki variables
        slc_mudf = var_mc_df.mu.pfp.trk.truth.p
        slc_pdf = var_mc_df.p.pfp.trk.truth.p
        slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
        slc_P_p_col = pad_column_name(("totp",), slc_pdf)
        tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
        for var_name in tki_var_names:
            var_mc_df = multicol_add(var_mc_df, tki_reco[var_name].rename("mc_" + var_name))

In [ ]:
for var_config in var_configs:

    ret = signal_hists(evtdf=ret_dfs["evt"], var_config=var_config, save_fig=False, plot=False, return_data=True)
    n_cv = ret["nevts_allsel_reco"]

    for widx, detvar_dfs in enumerate([ret_0xSCE_dfs, ret_2xSCE_dfs]):
        for k in detvar_dfs.keys():
            if "meta" in k:
                continue

            var_mc_df = detvar_dfs[k]
            var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
            var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)

            ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
            n_var = ret["nevts_allsel_reco"]
            n_var, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_var, 
                                histtype="step", color=f"C{widx}", alpha=0.3, linewidth=2)

    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_cv, histtype="step", color="black", label="CV")

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Events / Bin")

    plt.hist([], histtype="step", color="C0", label="0xSCE")
    plt.hist([], histtype="step", color="C1", label="2xSCE")
    plt.legend(frameon=False, loc="best")

    if save_fig:
        plt.savefig(f"{save_fig_dir}/{var_config.var_save_name}_SCEvar_evt_rates.png", bbox_inches='tight', dpi=300)
    plt.show()

    # get uncertainty
    wiremod_titles = ["0xSCE", "2xSCE"]
    wiremod_tags = ["0xSCE", "2xSCE"]
    tot_frac_unc = np.zeros(len(var_config.bin_centers))
    for widx, detvar_dfs in enumerate([ret_0xSCE_dfs, ret_2xSCE_dfs]):

        this_var_univ = []
        for k in detvar_dfs.keys():
            if "meta" in k:
                continue

            var_mc_df = detvar_dfs[k]
            var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
            var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)

            ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
            n_var = ret["nevts_allsel_reco"]
            this_var_univ.append(n_var)

        var_diffs = [this_var_univ[i] - n_cv for i in range(len(this_var_univ))]

        max_arg_diff = np.zeros(len(var_config.bin_centers))
        for i in range(len(var_config.bin_centers)):
            this_element_vals = [np.abs(v[i]) for v in var_diffs]
            max_arg = np.argmax(this_element_vals)
            max_arg_diff[i] = var_diffs[max_arg][i] 
        max_arg_diff

        n_var = n_cv + max_arg_diff
        ret = get_covariance_matrix(np.array([n_var]), n_cv)

        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        tot_frac_unc = tot_frac_unc + frac_unc**2

        plot_labels = ["", "", wiremod_titles[widx]]
        save_fig_name = "{}/{}-{}-{}".format(save_fig_dir, var_config.var_save_name, wiremod_tags[widx], "frac_unc")
        plot_frac_unc([frac_unc], var_config, plot_labels=plot_labels, save_fig=save_fig, save_name=save_fig_name)

        # matrix_type = "cov"
        # save_fig_name = "{}/{}-{}-{}".format(save_fig_dir, var_config.var_save_name, syst_type, matrix_type)
        # title = "{} {}".format(syst_type, matrix_type)
        # plot_heatmap(ret[matrix_type], 
        #             var_config.bins, 
        #             plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
        #             save_fig=save_fig, save_name=save_fig_name)

        detector_dict[f"detector-{wiremod_tags[widx]}"][var_config.var_save_name] = ret
    
    tot_frac_unc = np.sqrt(tot_frac_unc)
    print(tot_frac_unc)

In [ ]:
detector_dict

In [ ]:
print("saving dict with keys: ", detector_dict.keys())
print("for systs: ", detector_dict[list(detector_dict.keys())[0]].keys())
save_filename = f"{save_fig_dir}/{syst_type}_syst_dict.npz"
print("saving detector_dict as npz in %s" % (save_filename))
np.savez(save_filename, **detector_dict)

# Scratch

In [ ]:
detvar_tag = "ccal"
for var_config in [VariableConfig.chi2_mu(), VariableConfig.chi2_proton()]:

    with open(f'{sample_dir}/_{var_config.var_save_name}.pkl', 'rb') as f:
        ret_hist_chi2mu = pickle.load(f)
    nevts_mc = ret_hist_chi2mu['total_mc']
    n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                            alpha=0.3, color="gray", label="CV", density=False)

    with open(f'{sample_dir}/WireMod_XThetaXW_updatecalo_CV_{var_config.var_save_name}.pkl', 'rb') as f:
        ret_hist_chi2mu = pickle.load(f)
    nevts_mc = ret_hist_chi2mu['total_mc']
    n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                            histtype="step", linewidth=2, color="C0", density=False)

    with open(f'{sample_dir}/WireMod_YZ_updatecalo_CV_{var_config.var_save_name}.pkl', 'rb') as f:
        ret_hist_chi2mu = pickle.load(f)
    nevts_mc = ret_hist_chi2mu['total_mc']
    n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                            histtype="step", linewidth=2, color="C1", density=False)

    linestyles = ["--", ":"]
    for widx, wiremod_tag in enumerate(["XThetaXW", "YZ"]): #, "CV"]:
        for sidx, sign_tag in enumerate(["p", "m"]): # "CV"
                syst_tag = f"WireMod_{wiremod_tag}_updatecalo_{detvar_tag}_{sign_tag}"
                var_label = wiremod_tag + " & " + detvar_tag + "_" + sign_tag
                with open(f'{sample_dir}/{syst_tag}_{var_config.var_save_name}.pkl', 'rb') as f:
                    ret_hist_chi2mu = pickle.load(f)

                nevts_mc = ret_hist_chi2mu['total_mc']
                n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                                        histtype="step", linewidth=2, color=f"C{widx}", linestyle=linestyles[sidx], density=False)

    # plot data on top of MC (data points should be same value for all var files)
    nevts_data = ret_hist_chi2mu['total_data']
    err_data = np.sqrt(nevts_data)
    scale_factor = n.sum() / nevts_data.sum()
    plt.errorbar(var_config.bin_centers, nevts_data * scale_factor, yerr=err_data * scale_factor, fmt='o', label='Data', color='black')

    plt.hist([], linewidth=2, color="C0", histtype="step", label="WireMod X-$\\theta_{XW}$")
    plt.hist([], linewidth=2, color="C1", histtype="step", label="WireMod Y-Z")
    plt.hist([], linewidth=2, color="k",  histtype="step", linestyle="--", label=r"$C_{cal} + 1\sigma$")
    plt.hist([], linewidth=2, color="k",  histtype="step", linestyle=":", label=r"$C_{cal} - 1\sigma$")

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_plot_name)
    plt.ylabel("Tracks / Bin")
    plt.legend(frameon=False, fontsize=12)

    if save_fig:
        plt.savefig(f"{save_fig_dir}/{var_config.var_save_name}_envelopes_{detvar_tag}.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
# interpolation between distributions
from pyanalib.stat_helpers import *

calo_param = "ccal"
env_tags = [f"WireMod_XThetaXW_updatecalo_{calo_param}_p", f"WireMod_YZ_updatecalo_{calo_param}_m"]
# calo_param = "alpha"
# env_tags = [f"WireMod_XThetaXW_updatecalo_{calo_param}_p", f"WireMod_YZ_updatecalo_{calo_param}_m"]
# calo_param = "beta"
# env_tags = [f"WireMod_XThetaXW_updatecalo_{calo_param}_m", f"WireMod_YZ_updatecalo_{calo_param}_p"]
# calo_param = "R"
# env_tags = [f"WireMod_XThetaXW_updatecalo_{calo_param}_p", f"WireMod_YZ_updatecalo_{calo_param}_p"]

nevts_env = []
# for syst_tag in env_tags:
#     with open(f'{sample_dir}/{syst_tag}_chi2_mu.pkl', 'rb') as f:
#         ret_hist_chi2mu = pickle.load(f)

#     n = ret_hist_chi2mu['total_mc']
#     nevts_env.append(n)

with open(f'{sample_dir}/_{var_config.var_save_name}.pkl', 'rb') as f:
    ret_hist_chi2mu = pickle.load(f)
nevts_mc = ret_hist_chi2mu['total_mc']
n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                        alpha=0.3, color="gray", label="CV", density=False)
nevts_env.append(n)

with open(f'{sample_dir}/WireMod_XThetaXW_updatecalo_CV_{var_config.var_save_name}.pkl', 'rb') as f:
    ret_hist_chi2mu = pickle.load(f)
nevts_mc = ret_hist_chi2mu['total_mc']
n, bins, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=nevts_mc, 
                        histtype="step", linewidth=2, color="C0", density=False)
nevts_env.append(n)

# plot data on top of MC (data points should be same value for all var files)
nevts_data = ret_hist_chi2mu['total_data']
err_data = np.sqrt(nevts_data)
plt.errorbar(var_config.bin_centers, nevts_data, yerr=err_data, fmt='o', label='Data', color='black')

chi2_list = []
p_val_list = []
for n in nevts_env:
    # data stat error is the covariance matrix
    cov = np.diag(err_data ** 2)
    chi2val = get_chi2(nevts_data, n, cov)
    chi2_list.append(chi2val[0])
    p_val_list.append(chi2val[1])

# interpolate between the two distributions to find the minimum chi2
def interpolate_nevts(n1, n2, alpha):
    return alpha * n1 + (1 - alpha) * n2

alpha_list = np.linspace(0, 1, 51)
min_chi2 = 1e8
min_alpha = 1e8
min_pval = 1e8
for alpha in alpha_list:
    n_this_interp, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=interpolate_nevts(nevts_env[0], nevts_env[1], alpha), 
                                    histtype="step", alpha=0.5, color="gray")
    chi2val = get_chi2(nevts_data, n_this_interp, cov)

    if chi2val[0]/ len(var_config.bin_centers) < min_chi2:
        min_chi2 = chi2val[0]/ len(var_config.bin_centers)
        min_pval = chi2val[1]
        min_alpha = alpha

for nidx, n in enumerate(nevts_env):
    # pop string "updatecalo" from env_tags[nidx]
    this_legend = env_tags[nidx].split("updatecalo_")[0] + env_tags[nidx].split("updatecalo_")[-1]
    n, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n, histtype="step", linewidth=2, 
    label=this_legend + r" ($\chi^2$ / ndof = {:.2f})".format(chi2_list[nidx]/ len(var_config.bin_centers), p_val_list[nidx]))

# plot best fit
min_chi2 =  min_chi2
p_val = min_pval

this_legend = r"Best fit ($\chi^2$ / ndof = {:.2f}, ".format(min_chi2) + r" p-value = {:.2f})".format(p_val)

n_best_interp, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=interpolate_nevts(nevts_env[0], nevts_env[1], min_alpha), 
                                histtype="step", linewidth=2, color="red", label=this_legend)

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_plot_name)
plt.ylabel("Tracks / Bin")
plt.legend(frameon=False, fontsize=10)

plt.savefig(f"{save_fig_dir}/{var_config.var_save_name}_{calo_param}_env_chi2.pdf", bbox_inches='tight')
print(f"{save_fig_dir}/{var_config.var_save_name}_{calo_param}_env_chi2.pdf")
plt.show()


In [ ]:
def add_opening_angle(df, nu=False):
    if nu:
        opening_angle = (
            df.mc.mu.dir[["x", "y", "z"]].values * df.mc.p.dir[["x", "y", "z"]].values
        ).sum(axis=1)

    else:
        opening_angle = (
            df.mu.pfp.trk.dir[["x", "y", "z"]].values * df.p.pfp.trk.dir[["x", "y", "z"]].values
        ).sum(axis=1)

    df["theta_mu_p"] = np.arccos(opening_angle)
    return df

def add_track_direction(df):
    print("adding track direction")
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","x")] = df.mu.pfp.trk.truth.p.genp.x / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","y")] = df.mu.pfp.trk.truth.p.genp.y / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","z")] = df.mu.pfp.trk.truth.p.genp.z / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","x")] = df.p.pfp.trk.truth.p.genp.x / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","y")] = df.p.pfp.trk.truth.p.genp.y / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","z")] = df.p.pfp.trk.truth.p.genp.z / df.p.pfp.trk.truth.p.totp
    return df

tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

In [ ]:

syst_type = "detector"
evt_rates_dict = {}
detector_dict = {}
max_var_tags = {"ccal": ["WireMod_XThetaXW_updatecalo_ccal_p", "WireMod_YZ_updatecalo_ccal_m"],
                "alpha": ["WireMod_XThetaXW_updatecalo_alpha_p", "WireMod_YZ_updatecalo_alpha_m"],
                "beta": ["WireMod_XThetaXW_updatecalo_beta_m", "WireMod_YZ_updatecalo_beta_p"],
                "R": ["WireMod_XThetaXW_updatecalo_R_p", "WireMod_YZ_updatecalo_R_m"]} 

# detvar_tag = "ccal"
# for detvar_tag in max_var_tags.keys():
for var_config in var_configs:
    for detvar_tag in ["ccal", "beta"]:
        this_var_tags = max_var_tags[detvar_tag]
        this_var_univ = []

        with open(f'{sample_dir}/_sel_mup.pkl', 'rb') as f:
            var_mc_df = pickle.load(f)['mc_df']


        # updated df
        var_mc_df = add_opening_angle(var_mc_df)
        var_mc_df = add_track_direction(var_mc_df)

        slc_mudf = var_mc_df.mu.pfp.trk.truth.p
        slc_pdf = var_mc_df.p.pfp.trk.truth.p
        slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
        slc_P_p_col = pad_column_name(("totp",), slc_pdf)
        tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
        for var_name in tki_var_names:
            var_mc_df = multicol_add(var_mc_df, tki_reco[var_name].rename("mc_" + var_name))

        var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
        var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)
        ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
        n_var = ret["nevts_allsel_reco"]
        n_cv, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_var, histtype="step", linewidth=2, color="k", label="CV", density=False)

        for syst_tag in this_var_tags:
            var_label = syst_tag

            print(syst_tag)
            with open(f'{sample_dir}/{syst_tag}_sel_mup.pkl', 'rb') as f:
                var_mc_df = pickle.load(f)['mc_df']

            var_mc_df.loc[:,'topo_categ'] = get_topo_category(var_mc_df)
            var_mc_df.loc[:,'genie_categ'] = get_genie_category(var_mc_df)

            ret = signal_hists(evtdf=var_mc_df, var_config=var_config, save_fig=False, plot=False, return_data=True)
            n_var = ret["nevts_allsel_reco"]
            n_var, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_var, histtype="step", linewidth=2, density=False)

            # evt_rates_dict[var_config.var_save_name][syst_tag] = np.abs((n_var / n_cv) - 1)
            # print(n_var/n_cv)
            this_var_univ.append(n_var)

        plt.xlim(var_config.bins[0], var_config.bins[-1])
        plt.xlabel(var_config.var_labels[1])
        plt.ylabel("Events / Bin")
        plt.legend(frameon=False)

        # if save_fig:
        #     plt.savefig(f"{save_fig_dir}/{var_config.var_save_name}_detvar_evt_rates.pdf", bbox_inches='tight')
        # plt.show()

        cv_events = n_cv
        this_unisim = np.max(np.abs(np.array(this_var_univ)), axis=0)
        print(cv_events)
        print(this_unisim)
        univ_events = np.array([this_unisim])

        ret = get_covariance_matrix(univ_events, cv_events)


        plot_univ_hists(univ_events, cv_events, syst_type, var_config)

        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        plot_frac_unc([frac_unc], var_config)

        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_type, matrix_type)
        title = "{} {}".format(syst_type, matrix_type)
        plot_heatmap(ret[matrix_type], 
                    var_config.bins, 
                    plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                    save_fig=save_fig, save_name=save_fig_name)

        detector_dict[var_config.var_save_name] = {}
        detector_dict[var_config.var_save_name]["detector"] = ret

In [ ]:
detector_dict["detector"]["muon-p"]["cov_frac"]

# G4

In [ ]:
from makedf.g4syst import *
from pyanalib.covariance import *

# no uncertainty on kaons
g4_systematics = [
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4']

In [ ]:
save_fig = True
show_plot = True

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics_studies_G4-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)


# file with hist counts from running detvar samples through event selection
sample_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10/nevts"

In [ ]:
concat_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09",
    chunk_tags=generate_tags("aj"),
    df_tag="",
    keys2load=["hdr", "evt"],
    n_max_concat=10,
    sub_dir="MC",
    sample_dir="BNB_cosmics"
)
mc_hdr_df = concat_dfs['hdr']
mc_evt_df = concat_dfs['evt']

# ===== total pot =====
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = 1.0
mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))

for i in range(100):
    var = mc_evt_df.mc.G4["univ_{}".format(i)]
    if len(var[var > 1e3]) > 0:
        print(i)
        bad_idx = mc_evt_df[var > 1e3].index
        mc_evt_df = mc_evt_df.drop(bad_idx)
        print(f"dropped event with index {bad_idx}")

In [ ]:
mc_evt_df.mc.reinteractions_neutron_Geant4

In [ ]:
var_config = VariableConfig.muon_momentum()

for g4_knob in g4_systematics:
    print(g4_knob)
    syst_name = ("mc", g4_knob)

    n_univ = 100
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, None, var_config, syst_name, n_univ)
    ret = get_covariance_matrix(univ_events, cv_events)

    save_fig = False
    save_name = f"{save_fig_dir}/{var_config.var_save_name}-{syst_name[1]}_{cov_type}-univ_hists"
    plot_univ_hists(univ_events, cv_events, syst_name, var_config, plot=plot, save_fig=save_fig, save_name=save_name)

    matrix_type = "cov_frac"
    plot_labels = [var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"]
    save_name = f"{save_fig_dir}/{var_config.var_save_name}-{syst_name[1]}_{cov_type}-{matrix_type}"
    plot_heatmap(ret[matrix_type], 
                var_config.bins, 
                plot_labels=plot_labels,
                plot=True,
                save_fig=save_fig, save_name=save_name)

# Load all uncertainties, plot select knobs

In [ ]:
save_fig = True
show_plot = True

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics_studies_breakdown-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
def plot_syst_uncert(unc_dict, var_config, syst_names, systs, syst_on="xsec", save_fig=False, save_fig_name=None):

    fig, ax = plt.subplots()
    frac_uncert_total = np.zeros(len(var_config.bin_centers))

    # get top 5 unc on the integrated rate / xsec
    names = []
    vals = []
    labels = []
    for syst_name, syst in zip(syst_names, systs):
        syst_uncert = unc_dict[VariableConfig.all_events().var_save_name][syst][syst_on]
        if syst_on == "xsec":
            syst_uncert = syst_uncert
        frac_uncert_total += syst_uncert ** 2

        val = unc_dict["integrated"][syst][syst_on]
        names.append(syst)
        label_clean = syst_name.replace("GENIEReWeight_SBN_v1_multisim_", "")
        label_clean = label_clean.replace("GENIEReWeight_SBN_v1_multisigma_", "")
        label_clean = label_clean.replace("GENIEReWeight_SBNNuSyst_multisigma_", "")
        label_clean = label_clean.replace("CCQETemplateReweight_SBNNuSyst_multisigma", "")
        label_clean = label_clean.replace("ZExpPCAWeighter_SBNNuSyst_multisigma_", "")
        label_clean = label_clean.replace("QEInterference_SBNNuSyst_multisigma_INT_", "")
        label_clean = label_clean.replace("MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse", "")
        label_clean = label_clean.replace("EDep", "")
        labels.append(label_clean)
        vals.append(val[0])

    sorted_names = [x for _, x, _ in sorted(zip(vals, names, labels), key=lambda pair: pair[0])]
    sorted_labels = [x for _, _, x in sorted(zip(vals, names, labels), key=lambda pair: pair[0])]
    top5_syst = sorted_names[-10:][::-1]
    top5_labels = sorted_labels[-10:][::-1]
    for syst_name, syst in zip(top5_labels, top5_syst):
        syst_uncert = unc_dict[var_config.var_save_name][syst][syst_on]
        plt.hist(var_config.bin_centers, bins=var_config.bins, weights=syst_uncert,   histtype="step", linewidth=2, label=syst_name)

    # plot differential rate / xsec for chosen variable
    names = []
    vals = []
    labels = []
    for syst_name, syst in zip(syst_names, systs):
        syst_uncert = unc_dict[var_config.var_save_name][syst][syst_on]
        if syst_on == "xsec":
            syst_uncert = syst_uncert
        frac_uncert_total += syst_uncert ** 2

        val = unc_dict["integrated"][syst][syst_on]
        names.append(syst)
        label_clean = syst_name.replace("GENIEReWeight_SBN_v1_multisim_", "")
        label_clean = label_clean.replace("GENIEReWeight_SBN_v1_multisigma_", "")
        label_clean = label_clean.replace("GENIEReWeight_SBNNuSyst_multisigma_", "")
        label_clean = label_clean.replace("CCQETemplateReweight_SBNNuSyst_multisigma", "")
        label_clean = label_clean.replace("ZExpPCAWeighter_SBNNuSyst_multisigma_", "")
        label_clean = label_clean.replace("QEInterference_SBNNuSyst_multisigma_INT_", "")
        label_clean = label_clean.replace("MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse", "")
        labels.append(label_clean)
        vals.append(val[0])
        # plt.hist(var_config.bin_centers, bins=var_config.bins, weights=syst_uncert * 1e2,   histtype="step", linewidth=2, label=syst_name)


    frac_uncert_total = np.sqrt(frac_uncert_total)
    print(frac_uncert_total)
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total,    histtype="step", linewidth=2, color="k",  label="Total")

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.ylim(0, max(frac_uncert_total) * 1.5)

    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Uncertainty [%]")
    plt.legend(fontsize=10, ncol=3, loc="upper center")

    plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    plt.minorticks_on()

    if save_fig:
        plt.savefig(save_fig_name, bbox_inches='tight', dpi=300)
    plt.show();

In [ ]:
# cov matrix collector

all_unc_dicts = {}
base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
date_str = "20260216"

cov_mat_dict = {}
for var_config in var_configs:
    cov_mat_dict[var_config.var_save_name] = {
        "genie": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "flux": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "g4": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "det": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "mcstat": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
    }

In [ ]:
# GENIE

# Ar23
for genie_tag in ["CCQE", "MEC", "RES", "nonRES", "DIS", "Other"]:
    unc_file = path.join(base_dir, f"systematics-genie-{genie_tag}-{date_str}", f"genie-{genie_tag}_syst_dict.npz")
    unc_arr = np.load(unc_file, allow_pickle=True)
    unc_dict = dict(unc_arr)

    for var_config in var_configs:
        if var_config.var_save_name not in all_unc_dicts:
            all_unc_dicts[var_config.var_save_name] = {}
        syst_keys = list(unc_dict[var_config.var_save_name].item().keys())
        for k in syst_keys:
            syst_uncert_xsec = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]))
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]))
            all_unc_dicts[var_config.var_save_name][k] = {
                "xsec": syst_uncert_xsec,
                "rate": syst_uncert_rate
            }
            cov_mat_dict[var_config.var_save_name]["genie"] += unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]


# Ar23+
unc_file = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie-Ar23p_syst_dict.npz"
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    if var_config.var_save_name not in all_unc_dicts:
        all_unc_dicts[var_config.var_save_name] = {}

    groups = [
                "CCQETemplateReweight_SBNNuSyst_multisigma_SF", "CCQETemplateReweight_SBNNuSyst_multisigma_CRPA",
                "QEInterference_SBNNuSyst_multisigma_INT_QEIntf", "MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse",
                'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_MFP_pi',
                'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrCEx_pi',
                'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrInel_pi',
                'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrAbs_pi',
                'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrPiProd_pi',
    ]

    for group in groups:
        group_syst_names = [k for k in ar23p_genie_systematics if group in k]
        cov_mat_dict[var_config.var_save_name][group] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
        cov_mat_dict[var_config.var_save_name][group+"_rate"] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))

        group_syst_uncert_xsec = np.zeros(len(var_config.bin_centers))
        group_syst_uncert_rate = np.zeros(len(var_config.bin_centers))

        for i, k in enumerate(group_syst_names):
            syst_uncert_xsec = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]))
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]))

            group_syst_uncert_xsec += syst_uncert_xsec
            group_syst_uncert_rate += syst_uncert_rate

            cov_mat_dict[var_config.var_save_name][group] += unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]
            cov_mat_dict[var_config.var_save_name][group+"_rate"] += unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]

        # print(group, group_syst_uncert_xsec)
        all_unc_dicts[var_config.var_save_name][group] = {
            "xsec": group_syst_uncert_xsec,
            "rate": group_syst_uncert_rate
        }

unc_file = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie-zexp_syst_dict.npz"
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    if var_config.var_save_name not in all_unc_dicts:
        all_unc_dicts[var_config.var_save_name] = {}

    group = 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp'
    group_syst_names = [k for k in ar23p_genie_systematics if group in k]
    cov_mat_dict[var_config.var_save_name][group] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
    cov_mat_dict[var_config.var_save_name][group+"_rate"] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))

    group_syst_uncert_xsec = np.zeros(len(var_config.bin_centers))
    group_syst_uncert_rate = np.zeros(len(var_config.bin_centers))

    for i, k in enumerate(group_syst_names):
        syst_uncert_xsec = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]))
        syst_uncert_rate = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]))

        group_syst_uncert_xsec += syst_uncert_xsec
        group_syst_uncert_rate += syst_uncert_rate

        cov_mat_dict[var_config.var_save_name][group] += unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]
        cov_mat_dict[var_config.var_save_name][group+"_rate"] += unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]

    # print(group, group_syst_uncert_xsec)
    all_unc_dicts[var_config.var_save_name][group] = {
        "xsec": group_syst_uncert_xsec,
        "rate": group_syst_uncert_rate
    }

In [ ]:
all_unc_dicts['integrated']['GENIEReWeight_SBN_v1_multisim_RPA_CCQE']

In [ ]:
# Flux

for bnb_tag in ["hadron", "xsec", "beam"]:
    unc_file = path.join(base_dir, f"systematics-flux-{bnb_tag}-{date_str}", f"flux-{bnb_tag}_syst_dict.npz")
    unc_arr = np.load(unc_file, allow_pickle=True)
    unc_dict = dict(unc_arr)

    for var_config in var_configs:
        # make subdict if not exists
        if var_config.var_save_name not in all_unc_dicts:
            all_unc_dicts[var_config.var_save_name] = {}
        syst_keys = list(unc_dict[var_config.var_save_name].item().keys())
        for k in syst_keys:
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]))
            all_unc_dicts[var_config.var_save_name][k] = {
                "rate": syst_uncert_rate
            }
            cov_mat_dict[var_config.var_save_name]["flux"] += unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]

In [ ]:
# G4

g4_tag = "G4"
date_str = "20260225"
unc_file = path.join(base_dir, f"systematics-{g4_tag}-{date_str}", f"{g4_tag}_syst_dict.npz")
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    # make subdict if not exists
    if var_config.var_save_name not in all_unc_dicts:
        all_unc_dicts[var_config.var_save_name] = {}
    syst_keys = list(unc_dict[var_config.var_save_name].item().keys())
    for k in syst_keys:
        this_cov = unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]
        syst_uncert_rate = np.sqrt(np.diag(this_cov))
        all_unc_dicts[var_config.var_save_name][k]["rate"] = syst_uncert_rate
        cov_mat_dict[var_config.var_save_name]["g4"] += this_cov

In [ ]:
# save cov_mat_dict
import pickle

print("base_dir: ", base_dir)
with open(path.join(base_dir, f"cov_mat_dict-{date_str}.pkl"), "wb") as f:
    pickle.dump(cov_mat_dict, f)

In [ ]:
all_unc_dicts["integrated"].keys()

In [ ]:
save_fig_dir

In [ ]:
# qe_names = ['CCQE RPA','CCQE Coulomb Corr.']
# mec_names = ["CCMEC Norm.", "NCMEC Norm.", "MEC Decay Angle"]
qe_names = qe_genie_systematics
mec_names = mec_genie_systematics
res_names = res_genie_systematics
nonres_names = nonres_genie_systematics
dis_names = dis_genie_systematics
other_names = ['MFP_pi', 'FrCEx_pi', 'FrInel_pi', 'FrAbs_pi', 'FrPiProd_pi', 'MFP_N', 'FrCEx_N', 'FrInel_N', 'FrAbs_N', 'FrPiProd_N', "NormCCCOH", "NormNCCOH", 'MaNCEL', 'EtaNCEL']

# ar23p_genie_systematics = [
#     # "ZExpPCAWeighter_SBNNuSyst_multisigma_D", #, "ZExpPCAWeighter_SBNNuSyst_multisigma_MvA",
#     "CCQETemplateReweight_SBNNuSyst_multisigma_SF", "CCQETemplateReweight_SBNNuSyst_multisigma_CRPA",
#     "QEInterference_SBNNuSyst_multisigma_INT_QEIntf", "MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse",
#     'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_VecFFCCQEshape', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_CoulombCCQE', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormCCMEC', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormNCMEC', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_DecayAngMEC',
#     'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_MFP_pi', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrCEx_pi', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrInel_pi', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrAbs_pi', 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrPiProd_pi',
# ]
# ar23p_name = [
#     "z-expansion", #, "ZExpPCAWeighter_SBNNuSyst_multisigma_MvA",
#     "SF reweight", "CRPA reweight",
#     "QE int.", "MEC reweight",
#     'EDepFSI_VecFFCCQEshape', 'EDepFSI_CoulombCCQE', 'EDepFSI_NormCCMEC', 'EDepFSI_NormNCMEC', 'EDepFSI_DecayAngMEC',
#     'EDepFSI_MFP_pi', 'EDepFSI_FrCEx_pi', 'EDepFSI_FrInel_pi', 'EDepFSI_FrAbs_pi', 'EDepFSI_FrPiProd_pi',
# ]

systs = qe_genie_systematics + mec_genie_systematics + res_genie_systematics + nonres_genie_systematics + dis_genie_systematics + groups + ['ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp']
syst_names = qe_names + mec_names + res_names + nonres_names + dis_names + groups + ['ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp']

# systs = qe_genie_systematics 
# syst_names = qe_names

# systs = mec_genie_systematics
# syst_names = mec_names

# systs = res_genie_systematics
# syst_names = res_names

# systs = nonres_genie_systematics
# syst_names = nonres_names

# systs = dis_genie_systematics
# syst_names = dis_names

# systs = other_genie_systematics
# syst_names = other_names

# systs = ar23p_genie_systematics
# syst_names = ar23p_name

# systs = [
#     "ZExpPCAWeighter_SBNNuSyst_multisigma_D",
#     "QEInterference_SBNNuSyst_multisigma_INT_QEIntf", "MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse",
# ]
# syst_names = [
#     "z-expansion",
#     "QE int.", "MEC reweight",
# ]

for var_config in var_configs:
    save_fig_name = "{}/genie_breakdown-{}-rate.pdf".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(all_unc_dicts, var_config, syst_names, systs, syst_on="rate", save_fig=True, save_fig_name=save_fig_name)
    save_fig_name = "{}/genie_breakdown-{}-xsec.pdf".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(all_unc_dicts, var_config, syst_names, systs, syst_on="xsec", save_fig=True, save_fig_name=save_fig_name)

In [ ]:
from makedf.bnbsyst import *

xsec_systs = bnb_systematics_xsec
xsec_syst_names = [r'$\pi$ inelastic', r'$\pi$ QE', r'$\pi$ total',
              r'N inelastic', r'N QE', r'N total']

hadron_systs = bnb_systematics_hadron
hadron_syst_names = [r'$K^{-}$ prod.', r'$K^{+}$ prod.', r'$K^{0}$ prod.',
              r'$\pi^{-}$ prod.', r'$\pi^{+}$ prod.']

beam_systs = bnb_systematics_beam
beam_syst_names = ["Skin depth", "Horn current"]

systs = xsec_systs + hadron_systs + beam_systs
syst_names = xsec_syst_names + hadron_syst_names + beam_syst_names

for var_config in var_configs:
    save_fig_name = "{}/flux_breakdown-{}.png".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(all_unc_dicts, var_config, syst_names, systs, syst_on="rate", save_fig=True, save_fig_name=save_fig_name)

In [ ]:
# from makedf.g4syst import *
# syst_names = [r'$K^{-}$', r'$K^{+}$', r'$n$', r'$\pi^{-}$', r'$\pi^{+}$', r'$p$']

g4_systematics = [
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4']

systs = g4_systematics
syst_names = [r'$n$', r'$\pi^{-}$', r'$\pi^{+}$', r'$p$']

for var_config in var_configs:
    save_fig_name = "{}/g4_breakdown-{}.pdf".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(all_unc_dicts, var_config, syst_names, systs, syst_on="rate", save_fig=True, save_fig_name=save_fig_name)

# Demo plots for technote

## Event rate variations for each source

In [ ]:
# makes weights for nu events only (i.e. cosmics won't have genie weights)
syst_knob_list = regen_systematics_sbnd_morph
for this_syst in tqdm(syst_knob_list):
    has_genie_weights = (mc_evt_df[this_syst].morph.notna())
    weights_morph = mc_evt_df[has_genie_weights][this_syst].morph

    var = mc_evt_df[has_genie_weights][var_config.var_evt_reco_col]
    var = np.clip(var, var_config.bins[0], var_config.bins[-1] - eps)

    weights_std = (weights_morph-1)
    rand_wgt = np.random.normal(loc=0, scale=1, size=(100))
    n_univ_list = []
    binned_n_univ_list = []
    for i in range(100):
        weights_rand = 1 + np.abs(rand_wgt[i]) * weights_std
        n_univ, _ = np.histogram(var, bins=var_config.bins, weights=weights_rand)
        n_univ_list.append(n_univ)

        n_univ_binned, _, _ = plt.hist(var_config.bin_centers, bins=var_config.bins, weights=n_univ, histtype="step", color="gray", alpha=0.5)
        binned_n_univ_list.append(n_univ_binned)

    alternate_cv_morph, _, _ = plt.hist(var, bins=var_config.bins, weights=weights_morph, histtype="step", color="steelblue")

    std_binned = np.std(binned_n_univ_list, axis=0)
    mean_binned = np.mean(binned_n_univ_list, axis=0)
    plt.errorbar(var_config.bin_centers, mean_binned, yerr=std_binned, fmt="o", color="red")

    plt.title(this_syst)
    plt.show()

## GENIE variations for each mode
- demonstrates why the GENIE uncertainty for this channel is small, by comparing the signal vs. background scale

In [ ]:
syst_name = "GENIE"
for midx,this_mc_evt_df in enumerate(mc_evt_df_divided):
    print(topology_list[midx])
    for uidx in range(99):
        univ_col_evt = (syst_name, "univ_{}".format(uidx), "", "", "", "", "", "")
        univ_col_mc = (syst_name, "univ_{}".format(uidx), "")
        weights = this_mc_evt_df[univ_col_evt].copy()
        weights[np.isnan(weights)] = 1 ## IMPORTANT: make nan weights to 1. to ignore them
        this_var = this_mc_evt_df[var_config.var_evt_reco_col]
        this_var = np.clip(this_var, var_config.bins[0], var_config.bins[-1] - eps)
        # background_univ, _ = np.histogram(this_var, bins=var_config.bins, weights=weights)
        if uidx == 0:
            background_univ, _, _ = plt.hist(this_var, bins=var_config.bins, weights=weights, histtype="step", color=topology_colors[midx], alpha=0.5, label="UV")
        else:
            background_univ, _, _ = plt.hist(this_var, bins=var_config.bins, weights=weights, histtype="step", color=topology_colors[midx], alpha=0.5)
        background_univ = np.array(background_univ)
    # background_cv, _ = np.histogram(this_var, bins=var_config.bins)
    background_cv, _, _ = plt.hist(this_var, bins=var_config.bins, histtype="step", color="black", label="CV")
    background_cv = np.array(background_cv)
    plt.title(topology_labels[midx])
    plt.show();

## Variations on eff, smearing

In [ ]:
# change response matrix to get just the smearing, for demo plots
def get_response_matrix_noeff(reco_vs_true, eff, bins, 
                     plot=True, var_labels=None,
                     save_fig=False, save_fig_name=None):
    denom = reco_vs_true.T.sum(axis=0)
    num = reco_vs_true.T
    Response = np.divide(
        # num * eff, denom,
        num, denom,
        out=np.zeros_like(num, dtype=float),  # fill with 0 where invalid
        where=denom != 0
    )

    if plot:
        fig, ax = plt.subplots(figsize=(10, 10))
        unif_bin = np.linspace(0., float(len(bins) - 1), len(bins))
        extent = [unif_bin[0], unif_bin[-1], unif_bin[0], unif_bin[-1]]
        plt.imshow(Response, extent=extent, origin="lower", cmap="viridis")
        plt.colorbar(label="Response", shrink=0.7)

        x_edges = np.array(bins)
        y_edges = np.array(bins)
        x_tick_positions = (unif_bin[:-1] + unif_bin[1:]) / 2
        y_tick_positions = (unif_bin[:-1] + unif_bin[1:]) / 2

        x_labels = bin_range_labels(x_edges)
        y_labels = bin_range_labels(y_edges)

        plt.xticks(x_tick_positions, x_labels, rotation=45, ha="right")
        plt.yticks(y_tick_positions, y_labels)

        if var_labels is not None:
            plt.xlabel(var_labels[2], fontsize=20)
            plt.ylabel(var_labels[1], fontsize=20)

        for i in range(Response.shape[0]):      # rows (y)
            for j in range(Response.shape[1]):  # columns (x)
                value = Response[i, j]
                if not np.isnan(value):  # skip NaNs
                    plt.text(
                        j + 0.5, i + 0.5,
                        f"{value:.3f}",
                        ha="center", va="center",
                        color=get_text_color(value),
                        fontsize=10
                    )

        plt.title("Response", fontsize=20)
        plt.tight_layout()

        if save_fig and save_fig_name is not None:
            plt.savefig("{}.pdf".format(save_fig_name), bbox_inches="tight")
        plt.show()
    return Response

In [ ]:
from analysis_village.unfolding.unfolding_inputs import get_text_color, bin_range_labels

true_signal_univ, _ = np.histogram(var_truth_signal, bins=var_config.bins)
smear_cv = get_smear_matrix(var_signal_sel_truth, var_signal_sel_reco, var_config.bins, plot=False)
eff_cv = get_eff(smear_cv, true_signal_univ) 

eff_vars = ret_genie["univ_effs"]
for i in range(len(eff_vars)):
    if i == 0:
        plt.hist(var_config.bin_centers, bins=var_config.bins, weights=eff_vars[i], histtype="step", color="steelblue", alpha=0.5,
        label="UV")
    else:
        plt.hist(var_config.bin_centers, bins=var_config.bins, weights=eff_vars[i], histtype="step", color="steelblue", alpha=0.5)
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=eff_cv, histtype="step", color="black", label="CV")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Efficiency")
plt.title("GENIE Systematics")
plt.legend()
if save_fig:
    plt.savefig(path.join(save_fig_dir, "{}-{}-genie-effs".format(var_config.var_save_name, syst_name)))
plt.show();


smear_vars = ret_genie["univ_smears"]
smear_std = np.std(smear_vars, axis=0)
smear_mean = np.mean(smear_vars, axis=0)

# diagonal elements
for i in range(len(smear_vars)):
    this_diag = np.diag(smear_vars[i])
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=this_diag, histtype="step", color="steelblue", alpha=0.5)
cv_diag = np.diag(smear_cv)
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=cv_diag, histtype="step", color="black")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Diagonal Elements of Smearing Matrix")
plt.title("GENIE Systematics")
plt.show();

# full matrix
fig, ax = plt.subplots(figsize=(10, 10))
unif_bin = np.linspace(0., float(len(var_config.bins) - 1), len(var_config.bins))
extent = [unif_bin[0], unif_bin[-1], unif_bin[0], unif_bin[-1]]

plt.imshow(smear_std/smear_cv, extent=extent, origin="lower", cmap="viridis")
plt.colorbar(label="Univ Std / Univ Mean", shrink=0.7)

x_edges = np.array(var_config.bins)
y_edges = np.array(var_config.bins)
x_tick_positions = (unif_bin[:-1] + unif_bin[1:]) / 2
y_tick_positions = (unif_bin[:-1] + unif_bin[1:]) / 2

x_labels = bin_range_labels(x_edges)
y_labels = bin_range_labels(y_edges)

plt.xticks(x_tick_positions, x_labels, rotation=45, ha="right")
plt.yticks(y_tick_positions, y_labels)

for i in range(smear_std.shape[0]):      # rows (y)
    for j in range(smear_std.shape[1]):  # columns (x)
        value = smear_std[i, j] / smear_cv[i, j]
        if not np.isnan(value):  # skip NaNs
            plt.text(
                j + 0.5, i + 0.5,
                f"{value:.2f}",
                ha="center", va="center",
                color=get_text_color(value),
                fontsize=10
            )

plt.xlabel(var_config.var_labels[2])
plt.ylabel(var_config.var_labels[1])
plt.title("GENIE Systematics")
plt.show();

# smeared event rates
for i in range(len(smear_vars)):
    # this_events = smear_vars[i] @ nevts_signal_truth
    this_response = get_response_matrix_noeff(smear_vars[i], eff_vars[i], var_config.bins, plot=False)
    this_events = this_response @ nevts_signal_sel_truth
    if i == 0:
        plt.hist(var_config.bin_centers, bins=var_config.bins, weights=this_events, histtype="step", color="steelblue", alpha=0.5,
        label="UV")
    else:
        plt.hist(var_config.bin_centers, bins=var_config.bins, weights=this_events, histtype="step", color="steelblue", alpha=0.5)
# cv_events = smear_cv @ nevts_signal_sel_truth # note that we multiply the CV signal rate!
cv_response = get_response_matrix_noeff(smear_cv, eff_cv, var_config.bins, plot=False)
cv_events = cv_response @ nevts_signal_sel_truth
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=cv_events, histtype="step", color="black", label="CV")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Selected Event Rates")
plt.title("GENIE Systematics")
plt.legend()
if save_fig:
    plt.savefig(path.join(save_fig_dir, "{}-{}-genie-smeared_events".format(var_config.var_save_name, syst_name)))
plt.show();